# Passaggio 7: metriche e controlli di robustezza di tutte le varianti (solo train)

**Domanda.** Le differenze tra le varianti e il loro riferimento del Passaggio 6 ([06_backtest.ipynb](06_backtest.ipynb)) sono vere o fortuna? Per ogni variante:

1. tabella finale: metriche complete, differenza dal riferimento con intervallo di confidenza (bootstrap), beta e alpha rispetto a SPY;
2. sensibilità ai costi;
3. dipendenza da un settore o da una coppia;
4. dipendenza dal periodo: le due metà del train e il train senza il 13 agosto (il giorno di CRM-WDAY).

Scenario principale: costo 0. **Solo train**: il test non è stato letto.

**Cosa non si fa qui.** La sensibilità ai parametri vicini del segnale (soglie, emivita) è nel Passaggio 4 per la A. Per le varianti non si può fare senza rifare girare gli agenti, perché le loro decisioni valgono solo per i segnali con i parametri scelti.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # cartella del progetto: rende importabile src

import pandas as pd

from src.config import RESULTS
from src.backtest import load_trades, trading_days, add_costs, daily_pnl, summary, save_results, CAPITAL_PER_PAIR
from src.varianti_agente.confronto import riferimenti, tutte_le_varianti, delta_giornaliero, bootstrap, mercato, beta_alpha, settori

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
trades, days = load_trades(), trading_days()          # solo train
rif = riferimenti(trades)
varianti = tutte_le_varianti(trades)
mkt = mercato(days)
key = lambda v: (v.nome, v.versione)                  # noqa: E731
print(f"{len(varianti)} varianti, {len(days)} giorni di train; SPY: rendimento medio {mkt.mean() * 252:.1%} annuo, volatilità {mkt.std() * 252 ** 0.5:.1%}")

21 varianti, 64 giorni di train; SPY: rendimento medio 1.3% annuo, volatilità 11.7%


## 1. Tabella finale (costo 0)

- metriche di ogni variante e dei riferimenti;
- `delta_usd`: differenza in dollari dal riferimento, con l'intervallo di confidenza al 95% ottenuto ricampionando i 64 giorni con reinserimento (10.000 volte);
- `quota_sotto_zero`: la quota di ricampionamenti in cui la variante guadagna meno del riferimento;
- beta e alpha annualizzato rispetto a SPY, con le loro statistiche t. SPY: prezzi giornalieri aggiustati per i dividendi, scaricati da Alpaca (`python -m src.spy`).

Il bootstrap tratta i giorni come indipendenti.

In [2]:
def metrics(t):
    s = summary(add_costs(t, 0), days)
    return {k: s[k] for k in ("trade", "net_bps", "win_rate", "profit_factor", "avg_minutes", "pnl_usd", "sharpe")} | dict(max_drawdown_pct=100 * s["max_drawdown"]) | \
           beta_alpha(daily_pnl(add_costs(t, 0), days).ret, mkt)

rows = [dict(variante=n, versione="", riferimento="") | metrics(t) for n, t in rif.items()]
rows += [dict(variante=v.nome, versione=v.versione, riferimento=v.riferimento) | metrics(v.trades) | bootstrap(delta_giornaliero(v, rif, days))
         for v in varianti]
finale = pd.DataFrame(rows).set_index(["variante", "versione"])
finale = pd.concat([finale.iloc[:len(rif)], finale.iloc[len(rif):].sort_values("delta_usd", ascending=False)])
finale[["riferimento", "trade", "net_bps", "win_rate", "profit_factor", "sharpe", "max_drawdown_pct", "pnl_usd", "delta_usd", "ic95_basso", "ic95_alto",
        "quota_sotto_zero", "beta", "t_beta", "alpha_annuo", "t_alpha"]]

,,riferimento,trade,net_bps,win_rate,profit_factor,sharpe,max_drawdown_pct,pnl_usd,delta_usd,ic95_basso,ic95_alto,quota_sotto_zero,beta,t_beta,alpha_annuo,t_alpha
variante,versione,,,,,,,,,,,,,,,,
A,,,973,7.44,0.60,1.45,8.13,-0.15,"7,239.43",NaN,NaN,NaN,NaN,0.01,0.59,0.09,4.07
A_5min,,,964,5.26,0.59,1.30,6.08,-0.16,"5,071.41",NaN,NaN,NaN,NaN,-0.00,-0.30,0.07,3.05
A_15min,,,941,7.18,0.63,1.48,6.75,-0.14,"6,751.98",NaN,NaN,NaN,NaN,-0.00,-0.08,0.09,3.38
b_esagerazione,B,A,938,8.04,0.61,1.50,8.30,-0.16,"7,539.24",299.81,-203.92,863.07,0.13,0.01,0.63,0.10,4.16
c_fresca,C,A,973,7.44,0.60,1.45,8.13,-0.15,"7,239.43",0.00,-0.00,0.00,0.00,0.01,0.59,0.09,4.07
finbert_ritardo_15min,B,A_15min,935,7.15,0.64,1.49,6.85,-0.14,"6,689.12",-62.86,"-1,086.38",745.11,0.49,-0.00,-0.07,0.09,3.42
finbert_ritardo_5min,B,A_5min,960,5.20,0.59,1.31,6.01,-0.16,"4,996.14",-75.27,-959.24,809.18,0.60,-0.00,-0.25,0.07,3.01
finbert_episodio,B,A,968,7.18,0.60,1.44,8.13,-0.14,"6,954.50",-284.92,"-1,598.48",834.27,0.64,0.01,0.62,0.09,4.07
3b_mezzo,B ridotta,A,973,7.14,0.60,1.45,7.93,-0.13,"6,817.60",-421.83,-911.54,90.56,0.95,0.01,0.78,0.09,3.98


## 2. Sensibilità ai costi

Differenza in dollari dal riferimento con 0, 2, 5 e 10 punti base per operazione. Ogni trade bloccato risparmia i costi: con costi alti anche un blocco a caso migliora il risultato, quindi un vantaggio che compare solo a costo alto non vuol dire che l'agente scelga bene.

In [3]:
COSTS = (0, 2, 5, 10)
costi = pd.DataFrame({key(v): {c: summary(add_costs(v.trades, c), days)["pnl_usd"] - summary(add_costs(rif[v.riferimento], c), days)["pnl_usd"]
                               for c in COSTS} for v in varianti}).T.rename_axis(["variante", "versione"])
costi.columns = [f"delta_usd_costo_{c}" for c in COSTS]
costi.sort_values("delta_usd_costo_0", ascending=False)

,,delta_usd_costo_0,delta_usd_costo_2,delta_usd_costo_5,delta_usd_costo_10
variante,versione,,,,
b_esagerazione,B,299.81,579.81,999.81,"1,699.81"
c_fresca,C,-0.00,0.00,0.00,0.00
finbert_ritardo_15min,B,-62.86,-14.86,57.14,177.14
finbert_ritardo_5min,B,-75.27,-43.27,4.73,84.73
finbert_episodio,B,-284.92,-244.92,-184.92,-84.92
3b_mezzo,B ridotta,-421.83,-281.83,-71.83,278.17
finbert_seduta,B,-469.07,-397.07,-289.07,-109.07
3b_6h,B,-499.91,-379.91,-199.91,100.09
finbert_episodio,C,-650.77,-618.77,-570.77,-490.77


## 3. Dipende da un settore o da una coppia?

Differenza in dollari dal riferimento (costo 0) per settore. Poi, per ogni variante, la coppia che pesa di più sulla differenza e la differenza senza quella coppia.

In [4]:
sector = settori()

def delta_by_pair(v):
    p = lambda t: add_costs(t, 0).groupby("pair").pnl_usd.sum()   # noqa: E731
    return p(v.trades).sub(p(rif[v.riferimento]), fill_value=0).reindex(sector.index, fill_value=0)

by_pair = pd.DataFrame({key(v): delta_by_pair(v) for v in varianti}).T.rename_axis(["variante", "versione"])
by_pair.T.groupby(sector).sum().T.loc[costi.sort_values("delta_usd_costo_0", ascending=False).index]

,gruppo,"Alimenti, bevande e tabacco",Attrezzature e servizi sanitar,Banche,Beni discrezionali Distribuzio,Energia,"Farmaceutica, Biotecnologie e",Prodotti per la casa e l'igien,Semiconduttori e apparecchiatu,Servizi finanziari,Software e servizi,Trasporto
variante,versione,,,,,,,,,,,
b_esagerazione,B,138.33,-41.77,135.37,10.76,-57.89,55.92,131.63,0.00,-118.68,46.13,0.00
c_fresca,C,-0.00,0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00
finbert_ritardo_15min,B,0.00,102.48,0.00,0.00,24.53,0.00,0.00,0.00,-404.57,214.71,0.00
finbert_ritardo_5min,B,0.00,135.81,0.00,0.00,-37.88,0.00,0.00,0.00,-455.56,282.36,0.00
finbert_episodio,B,0.00,31.05,0.00,0.00,-58.74,0.00,0.00,0.00,-554.90,297.67,0.00
3b_mezzo,B ridotta,-20.37,-194.99,-47.38,-4.21,-57.28,-44.15,0.49,0.00,-169.47,115.53,0.00
finbert_seduta,B,0.00,-93.76,36.21,0.00,-58.74,0.00,0.00,0.00,-554.90,297.67,-95.54
3b_6h,B,0.00,-385.05,-129.69,0.00,-55.83,-83.15,0.00,0.00,-143.87,297.67,0.00
finbert_episodio,C,-66.57,-24.92,-26.12,0.00,358.44,-99.47,0.00,0.00,"-1,445.62",653.49,0.00


In [5]:
top = by_pair.abs().idxmax(axis=1)
pd.DataFrame({"delta_usd": by_pair.sum(axis=1), "coppia_che_pesa_di_piu": top,
              "delta_di_quella_coppia": [by_pair.loc[i, p] for i, p in top.items()],
              "delta_senza_quella_coppia": [by_pair.loc[i].drop(p).sum() for i, p in top.items()]}).sort_values("delta_usd", ascending=False)

,,delta_usd,coppia_che_pesa_di_piu,delta_di_quella_coppia,delta_senza_quella_coppia
variante,versione,,,,
b_esagerazione,B,299.81,KO-PEP,138.33,161.48
c_fresca,C,0.00,PYPL-XYZ,0.00,0.00
finbert_ritardo_15min,B,-62.86,PYPL-XYZ,-505.56,442.70
finbert_ritardo_5min,B,-75.27,PYPL-XYZ,-455.56,380.29
finbert_episodio,B,-284.92,PYPL-XYZ,-563.88,278.95
3b_mezzo,B ridotta,-421.83,ELV-UNH,-192.52,-229.31
finbert_seduta,B,-469.07,PYPL-XYZ,-563.88,94.81
3b_6h,B,-499.91,ELV-UNH,-385.05,-114.86
finbert_episodio,C,-650.77,PYPL-XYZ,"-1,482.55",831.78


## 4. Dipende dal periodo?

Differenza in dollari dal riferimento (costo 0) nelle due metà del train (32 giorni ciascuna) e su tutto il train senza il 13 agosto, il giorno del salto di WDAY (Passaggio 4).

In [6]:
half = days[len(days) // 2]
periodo = pd.DataFrame({key(v): (lambda d: {"prima_meta": d[d.index < half].sum(), "seconda_meta": d[d.index >= half].sum(),
                                             "senza_13_agosto": d.drop(pd.Timestamp("2026-08-13")).sum(), "tutto": d.sum()})(delta_giornaliero(v, rif, days))
                        for v in varianti}).T.rename_axis(["variante", "versione"])
periodo.sort_values("tutto", ascending=False)

,,prima_meta,seconda_meta,senza_13_agosto,tutto
variante,versione,,,,
b_esagerazione,B,121.40,178.41,236.96,299.81
c_fresca,C,0.00,0.00,0.00,0.00
finbert_ritardo_15min,B,-302.09,239.23,-62.86,-62.86
finbert_ritardo_5min,B,-319.75,244.47,-75.27,-75.27
finbert_episodio,B,-532.83,247.90,-284.92,-284.92
3b_mezzo,B ridotta,-358.23,-63.60,-421.83,-421.83
finbert_seduta,B,-592.16,123.09,-469.07,-469.07
3b_6h,B,-533.50,33.59,-499.91,-499.91
finbert_episodio,C,-881.38,230.61,-650.77,-650.77


## 5. Salvataggio

In [7]:
for p in save_results({"robustezza/tabella_finale": finale, "robustezza/costi": costi, "robustezza/coppie": by_pair,
                       "robustezza/settori": by_pair.T.groupby(sector).sum().T, "robustezza/periodo": periodo}):
    print(p.relative_to(RESULTS.parent))

Results/robustezza/tabella_finale.csv
Results/robustezza/costi.csv
Results/robustezza/coppie.csv
Results/robustezza/settori.csv
Results/robustezza/periodo.csv


## 6. Cosa ci dicono i controlli

**Nessuna differenza si distingue dal caso.**
- Per quasi tutte le varianti l'intervallo al 95% della differenza dal riferimento contiene lo zero.
- Le eccezioni sono la C di `lessico` (da -4.600 a -499 dollari) e `spark24h_size_agente` (da -5.740 a -1.220): sono peggiori della A con una certa sicurezza.
- Anche `b_esagerazione`, l'unica sopra la A, ha un intervallo da -204 a +863 dollari: è sotto la A nel 13% dei ricampionamenti.

**Neutrale al mercato.** Rispetto a SPY tutte le varianti hanno un beta praticamente nullo (da -0,01 a +0,01, statistica t sotto 1,2), come ci si aspetta dal pairs trading: il guadagno non viene dall'andamento della borsa. L'alpha annualizzato della A (9,5%, t = 4,1) va letto con cautela: sono 64 giorni, riportati a un anno.

**Costi.** Con costi alti molte B superano la A (`lessico` B +5.113 dollari a 10 bps) solo perché fanno meno trade. È un risparmio di costi, non una scelta migliore: le stesse B a costo 0 perdono.

**Coppie e periodo.** La classifica dipende da una o due coppie e da pochi giorni.
- **Varianti FinBERT:** la differenza viene quasi tutta da PYPL-XYZ. Senza quella coppia, `finbert_seduta` B passa da -469 a +95 dollari, `finbert_episodio` B da -285 a +279, e le B ritardate diventano positive (+380 e +443). Inoltre perdono nella prima metà del train e guadagnano nella seconda.
- **Varianti di Giulio:** pesa CRM-WDAY. Senza il 13 agosto, la B di `spark_24h` perde 395 dollari invece di 1.334.
- **`b_esagerazione`:** è sopra la A in entrambe le metà, ma ha la regola scelta su questi dati.

**Le 3 candidate del Passaggio 6.** Sono `finbert_ritardo_15min` B, `finbert_ritardo_5min` B e `finbert_episodio` B: la classifica e il motivo delle esclusioni sono nel Passaggio 6. I controlli qui sopra dicono:

| Candidata | Differenza dal riferimento | Intervallo al 95% | Senza PYPL-XYZ | Prima metà / seconda metà |
|---|---|---|---|---|
| `finbert_ritardo_15min` B | -63 | da -1.086 a +745 | +443 | -302 / +239 |
| `finbert_ritardo_5min` B | -75 | da -959 a +809 | +380 | -320 / +244 |
| `finbert_episodio` B | -285 | da -1.598 a +834 | +279 | -533 / +248 |

- **Nessuna si distingue dal suo riferimento:** tutti gli intervalli contengono lo zero.
- **Tutte e tre dipendono da PYPL-XYZ:** senza quella coppia diventano positive. La perdita viene dal blocco del 15 luglio su PYPL-XYZ, un trade che nella A guadagnava molto.
- **Tutte e tre perdono nella prima metà del train e guadagnano nella seconda.**
- **Il 13 agosto non pesa:** senza quel giorno la differenza non cambia.
- **Beta nullo e alpha come quello del riferimento.**

Il gruppo sceglie quindi tra varianti che sul train non si distinguono dal loro riferimento. La scelta va motivata anche con la regola: per esempio, la finestra dell'episodio guarda solo le notizie uscite mentre lo spread si allontanava. Va scritta nel registro prima di aprire il test.

**Limiti.**
- Il bootstrap tratta i giorni come indipendenti.
- La sensibilità ai parametri del segnale non si può fare per le varianti senza rifare girare gli agenti.
- Con 21 varianti confrontate, ci si aspetta per caso qualche differenza che sembra significativa.